In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path("/mnt/dlabscratch1/samaier/projects/SDPO-safety")
OUT = REPO / "outputs/pretrain_benchmarks"

RUNS = {
    "stage12_8k": OUT / "longctx_opsd_stage2_gsm8k_maxnew7680_latest2" / "combined_summary.csv",
    "think_32k": OUT / "longctx_opsd_stage2_math500_maxnew7680_latest2" / "combined_summary.csv",
}

dfs = []
for label, path in RUNS.items():
    df = pd.read_csv(path)
    df["run_label"] = label
    dfs.append(df)

summary = pd.concat(dfs, ignore_index=True)

summary["model_task"] = summary["checkpoint"] + " / " + summary["task"]
for c in ["pass@1", "pass@8", "pass@32", "hit_max_new_tokens_rate"]:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

token_cols = [
    "completion_tokens_mean", "completion_tokens_median",
    "completion_tokens_p90", "completion_tokens_p95", "completion_tokens_p99",
    "completion_tokens_max", "raw_completion_tokens_mean",
    "raw_completion_tokens_p99", "raw_completion_tokens_max",
]
for c in token_cols:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

display_cols = [
    "run_label", "checkpoint", "task", "num_examples", "num_samples",
    "num_fewshot", "max_new_tokens",
    "pass@1", "pass@8", "pass@32",
    "completion_tokens_mean", "completion_tokens_median",
    "completion_tokens_p90", "completion_tokens_p99",
    "hit_max_new_tokens_rate",
]
display(summary[display_cols].round(4))

,run_label,checkpoint,task,num_examples,num_samples,num_fewshot,max_new_tokens,pass@1,pass@8,pass@32,completion_tokens_mean,completion_tokens_median,completion_tokens_p90,completion_tokens_p99,hit_max_new_tokens_rate
0,stage12_8k,opsd_ema002_s2_47684_lr1e-6_latest,gsm8k,1319,32,0,7680,0.4152,0.9202,0.9765,326.2908,133,306,7680,0.0234
1,stage12_8k,opsd_ema0_safe_s2_47684_lr1e-6_latest,gsm8k,1319,32,0,7680,0.4486,0.9294,0.9795,264.0463,131,284,7680,0.0155
2,think_32k,opsd_ema002_s2_47684_lr1e-6_latest,math500,500,32,0,7680,0.3873,0.7611,0.8960,1570.7081,342,7680,7680,0.1410
3,think_32k,opsd_ema0_safe_s2_47684_lr1e-6_latest,math500,500,32,0,7680,0.3941,0.7530,0.8740,1476.9736,296,7680,7680,0.1342


In [3]:
from pathlib import Path
import json, random, html
from IPython.display import display, HTML

ROOT = Path("/dlabscratch1/samaier/projects/SDPO-safety")

# Pick one benchmark file. Use rescored files when available.
prediction_files = sorted(
    ROOT.glob("outputs/pretrain_benchmarks/longctx_opsd_stage2_gsm8k_maxnew7680*/**/*__gsm8k.rescored.jsonl")
)

prediction_files

[PosixPath('/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/longctx_opsd_stage2_gsm8k_maxnew7680/trained/opsd_ema0_safe_s2_32000_lr1e-6/20260601_222454/opsd_ema0_safe_s2_32000_lr1e-6__gsm8k.rescored.jsonl'),
 PosixPath('/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/longctx_opsd_stage2_gsm8k_maxnew7680/trained/opsd_ema0_safe_s2_32000_lr5e-6/20260602_013312/opsd_ema0_safe_s2_32000_lr5e-6__gsm8k.rescored.jsonl'),
 PosixPath('/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/longctx_opsd_stage2_gsm8k_maxnew7680/trained/opsd_ema0_safe_s2_47684_lr1e-6/20260602_035824/opsd_ema0_safe_s2_47684_lr1e-6__gsm8k.rescored.jsonl'),
 PosixPath('/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/longctx_opsd_stage2_gsm8k_maxnew7680_latest2/trained/opsd_ema002_s2_47684_lr1e-6_latest/20260601_124933/opsd_ema002_s2_47684_lr1e-6_latest__gsm8k.rescored.jsonl'),
 PosixPath('/dlabscratch1/samaier/projects/SDPO-safety/outpu

In [6]:
def render_generations(rows, max_samples=6, only=None, max_chars=2500):
    """
    only: None | correct | incorrect | recovered | format_wrong | hit_max_new_tokens
    """
    parts = []
    for row in rows:
        q = html.escape(str(row.get("question", "")))
        gt = html.escape(str(row.get("ground_truth", "")))
        checkpoint = html.escape(str(row.get("checkpoint", "")))
        idx = row.get("index", "")

        parts.append(f"""
        <div style="border:1px solid #9ca3af; border-radius:8px; padding:14px; margin:18px 0;
                    background:#111827; color:#f9fafb;">
          <div style="font-size:13px; color:#d1d5db;">
            <b>{checkpoint}</b> · example {idx} · gt: <b>{gt}</b> ·
            corrected_num_correct: {row.get("corrected_num_correct", row.get("num_correct", ""))}
          </div>
          <div style="margin-top:10px; font-size:16px; color:#ffffff;"><b>Question</b></div>
          <pre style="white-space:pre-wrap; background:#1f2937; color:#f9fafb;
                      padding:10px; border-radius:6px;">{q}</pre>
        """)

        samples = row.get("samples", [])
        if only == "correct":
            samples = [s for s in samples if s.get("correct", False)]
        elif only == "incorrect":
            samples = [s for s in samples if not s.get("correct", False)]
        elif only == "recovered":
            samples = [s for s in samples if s.get("correct", False) and not s.get("original_correct", False)]
        elif only == "format_wrong":
            samples = [s for s in samples if s.get("format_wrong", False)]
        elif only == "hit_max_new_tokens":
            samples = [s for s in samples if s.get("hit_max_new_tokens", False)]

        for s in samples[:max_samples]:
            correct = bool(s.get("correct", False))
            bg = "#052e16" if correct else "#450a0a"
            border = "#22c55e" if correct else "#ef4444"
            text = str(s.get("completion") or s.get("raw_completion") or "")
            if len(text) > max_chars:
                text = text[:max_chars] + "\n\n...[truncated]"

            extracted = html.escape(str(s.get("extracted_answer", "")))
            sample_index = s.get("sample_index", "")
            toks = s.get("completion_tokens", s.get("raw_completion_tokens", ""))

            parts.append(f"""
            <div style="border-left:5px solid {border}; background:{bg}; color:#f9fafb;
                        padding:10px; margin:10px 0; border-radius:6px;">
              <div style="font-size:13px; color:#e5e7eb;">
                <b>sample {sample_index}</b> · correct={correct} · extracted=<b>{extracted}</b> · tokens={toks}
              </div>
              <pre style="white-space:pre-wrap; margin-top:8px; color:#f9fafb;">{html.escape(text)}</pre>
            </div>
            """)

        parts.append("</div>")

    display(HTML("\n".join(parts)))


def load_generations(path, n_questions=5, seed=0, where="any"):
    """
    where: any | incorrect | correct | recovered | format_wrong | hit_max_new_tokens
    """
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            samples = row.get("samples", [])

            if where == "incorrect":
                keep = any(not s.get("correct", False) for s in samples)
            elif where == "correct":
                keep = any(s.get("correct", False) for s in samples)
            elif where == "recovered":
                keep = any(s.get("correct", False) and not s.get("original_correct", False) for s in samples)
            elif where == "format_wrong":
                keep = any(s.get("format_wrong", False) for s in samples)
            elif where == "hit_max_new_tokens":
                keep = any(s.get("hit_max_new_tokens", False) for s in samples)
            else:
                keep = True

            if keep:
                rows.append(row)

    rng = random.Random(seed)
    rng.shuffle(rows)
    return rows[:n_questions]

In [7]:
# path = prediction_files[0]
# rows = load_generations(path, n_questions=5, seed=3, where="incorrect")
# # render_generations(rows, max_samples=5, only="incorrect")
# render_generations(rows, max_samples=5, only=None)

path = prediction_files[2]
rows = load_generations(path, n_questions=5, seed=3, where="incorrect")
# render_generations(rows, max_samples=5, only="incorrect")
render_generations(rows, max_samples=5, only=None)

In [8]:
path = prediction_files[-1]
rows = load_generations(path, n_questions=5, seed=3, where="incorrect")
# render_generations(rows, max_samples=5, only="incorrect")
render_generations(rows, max_samples=5, only=None)

In [21]:
from dataclasses import dataclass
from pathlib import Path
import json, random, html
from IPython.display import display, HTML

@dataclass(frozen=True)
class GenerationModel:
    name: str
    path: Path


def read_jsonl(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def load_model_generations(models):
    loaded = []
    for model in models:
        rows = read_jsonl(model.path)
        by_question = {row["question"]: row for row in rows}
        loaded.append({"model": model, "rows": rows, "by_question": by_question})
    return loaded


def shared_model_questions(loaded_models):
    if not loaded_models:
        return []
    shared = set(loaded_models[0]["by_question"])
    for loaded in loaded_models[1:]:
        shared &= set(loaded["by_question"])
    return sorted(shared)


MODELS = [
    GenerationModel(
        "Base stage2 final",
        Path("/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/base_stage12_len8k_gsm8k_math_500/base/base_stage2_final/20260527_060000/base_stage2_final__gsm8k.jsonl"),
    ),
    GenerationModel(
        # "OPSD longctx 47684 lr1e-6",
        # Path("/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/longctx_opsd_stage2_gsm8k_maxnew7680/trained/opsd_ema0_safe_s2_47684_lr1e-6/20260602_035824/opsd_ema0_safe_s2_47684_lr1e-6__gsm8k.rescored.jsonl"),
        "OPSD longctx 47684 lr1e-6 latest",
        Path("/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/longctx_opsd_stage2_gsm8k_maxnew7680_latest2/trained/opsd_ema0_safe_s2_47684_lr1e-6_latest/20260601_084202/opsd_ema0_safe_s2_47684_lr1e-6_latest__gsm8k.rescored.jsonl"),
    ),
    GenerationModel(
        "GRPO stage2 47684",
        Path("/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/gsm8k_stage_all_h200g4/trained/grpo_stage2_47684/20260508_130639/grpo_stage2_47684__gsm8k.rescored.jsonl"),
    ),
]

loaded_models = load_model_generations(MODELS)
shared_questions = shared_model_questions(loaded_models)

[(m["model"].name, len(m["rows"])) for m in loaded_models], len(shared_questions)


([('Base stage2 final', 500),
  ('OPSD longctx 47684 lr1e-6 latest', 1319),
  ('GRPO stage2 47684', 1319)],
 500)

In [22]:
def sample_text(sample, max_chars=1800):
    text = str(sample.get("completion") or sample.get("raw_completion") or "")
    if len(text) > max_chars:
        text = text[:max_chars] + "\n\n...[truncated]"
    return html.escape(text)


def sample_matches(sample, only=None):
    if only in [None, "any"]:
        return True
    if only == "correct":
        return bool(sample.get("correct", False))
    if only == "incorrect":
        return not bool(sample.get("correct", False))
    if only == "recovered":
        return bool(sample.get("correct", False)) and not bool(sample.get("original_correct", False))
    if only == "format_wrong":
        return bool(sample.get("format_wrong", False))
    if only == "hit_max_new_tokens":
        return bool(sample.get("hit_max_new_tokens", False))
    raise ValueError(f"Unknown sample filter: {only}")


def pick_samples(row, only=None, max_samples=3):
    samples = [s for s in row.get("samples", []) if sample_matches(s, only=only)]
    return samples[:max_samples]


def row_matches(row, where=None):
    if where in [None, "any"]:
        return True
    return any(sample_matches(s, only=where) for s in row.get("samples", []))


def render_model_block(row, title, only=None, max_samples=3, max_chars=1800):
    samples = pick_samples(row, only=only, max_samples=max_samples)
    blocks = [f"""
    <div style="font-size:15px; font-weight:700; color:#f9fafb; margin-bottom:8px;">{html.escape(title)}</div>
    <div style="font-size:12px; color:#d1d5db; margin-bottom:8px;">
      checkpoint: {html.escape(str(row.get("checkpoint", "")))} ·
      num_correct: {row.get("corrected_num_correct", row.get("num_correct", ""))}
    </div>
    """]

    if not samples:
        blocks.append("""
        <div style="border-left:5px solid #6b7280; background:#1f2937; color:#d1d5db;
                    padding:10px; margin:10px 0; border-radius:6px;">
          No samples matched this filter.
        </div>
        """)
        return "\n".join(blocks)

    for s in samples:
        correct = bool(s.get("correct", False))
        bg = "#052e16" if correct else "#450a0a"
        border = "#22c55e" if correct else "#ef4444"
        extracted = html.escape(str(s.get("extracted_answer", "")))
        toks = s.get("completion_tokens", s.get("raw_completion_tokens", ""))
        blocks.append(f"""
        <div style="border-left:5px solid {border}; background:{bg}; color:#f9fafb;
                    padding:10px; margin:10px 0; border-radius:6px;">
          <div style="font-size:12px; color:#e5e7eb;">
            sample {s.get("sample_index", "")} · correct={correct} · extracted=<b>{extracted}</b> · tokens={toks}
          </div>
          <pre style="white-space:pre-wrap; color:#f9fafb; margin-top:8px;">{sample_text(s, max_chars=max_chars)}</pre>
        </div>
        """)
    return "\n".join(blocks)


def render_model_comparison(models=None, n=5, seed=0, where="any", only=None, max_samples=3, max_chars=1800):
    loaded = load_model_generations(models or MODELS)
    questions = shared_model_questions(loaded)

    if where not in [None, "any"]:
        questions = [
            q for q in questions
            if any(row_matches(loaded_model["by_question"][q], where=where) for loaded_model in loaded)
        ]

    rng = random.Random(seed)
    rng.shuffle(questions)
    questions = questions[:n]

    if not questions:
        display(HTML("<div style=\"color:#f9fafb; background:#111827; padding:12px;\">No shared questions matched.</div>"))
        return

    grid_cols = " ".join(["minmax(280px, 1fr)"] * len(loaded))
    parts = []
    for q in questions:
        first_row = loaded[0]["by_question"][q]
        gt = html.escape(str(first_row.get("ground_truth", "")))
        model_blocks = []
        for loaded_model in loaded:
            model = loaded_model["model"]
            row = loaded_model["by_question"][q]
            model_blocks.append(render_model_block(row, model.name, only=only, max_samples=max_samples, max_chars=max_chars))

        parts.append(f"""
        <div style="border:1px solid #6b7280; border-radius:8px; padding:14px; margin:18px 0;
                    background:#111827; color:#f9fafb;">
          <div style="font-size:13px; color:#d1d5db;">ground truth: <b>{gt}</b></div>
          <pre style="white-space:pre-wrap; background:#1f2937; color:#f9fafb;
                      padding:10px; border-radius:6px;">{html.escape(q)}</pre>
          <div style="display:grid; grid-template-columns:{grid_cols}; gap:14px; overflow-x:auto;">
            {"".join(f"<div>{block}</div>" for block in model_blocks)}
          </div>
        </div>
        """)

    display(HTML("\n".join(parts)))


In [23]:
render_model_comparison(MODELS, n=10, seed=4, where="any", only=None, max_samples=10)


In [15]:
GEN_PATH = "/dlabscratch1/samaier/projects/SDPO-safety/outputs/pretrain_benchmarks/gsm8k_stage_all_h200g4/trained/grpo_stage2_47684/20260508_130639/grpo_stage2_47684__gsm8k.rescored.jsonl"

rows = load_generations(GEN_PATH, n_questions=6, seed=0, where="any")
render_generations(rows, max_samples=1)